# 0 · Setup & smoke test

This notebook installs **langchain-engram** from source and verifies your
Engram credentials with a tiny add → wait → search round trip.

You need an Engram API key — create one at
<https://console.weaviate.cloud/engram>. No LLM key is required here.

## Setup — pick **one** option

**Option A — editable install (recommended).** Installs this package *and* its
dependencies from source; edits to `langchain_engram/` are picked up after a
kernel restart.

**Option B — import from local source (no install of `langchain-engram`).** Adds
the repo root to `sys.path` so `import langchain_engram` resolves straight to the
source tree. You still need the runtime dependencies available — the easiest is to
launch Jupyter from the repo's environment: `uv run --with jupyter jupyter lab`.

In [ ]:
# Option A — editable install from source.
%pip install -q -e ".."

In [ ]:
# Option B — use langchain-engram straight from the local source tree (no install).
# Run this INSTEAD of Option A.
import sys
from pathlib import Path

repo_root = Path.cwd().parent  # notebooks/ -> repo root (holds langchain_engram/)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import langchain_engram

print('langchain_engram imported from:', langchain_engram.__file__)

Provide your Engram API key (stored only in this kernel's environment):

In [ ]:
import getpass
import os

if not os.environ.get("ENGRAM_API_KEY"):
    os.environ["ENGRAM_API_KEY"] = getpass.getpass("ENGRAM_API_KEY: ")

# Optional: point at a non-default Engram endpoint.
# os.environ["ENGRAM_BASE_URL"] = "https://..."

The package should import and report its version:

In [ ]:
import langchain_engram

print('langchain-engram', langchain_engram.__version__)
print('exports:', langchain_engram.__all__)

### Round trip
Add a memory, wait for Engram's pipeline to commit it, then search it back.

In [ ]:
import time

from langchain_engram._client import build_client

client = build_client()  # reads ENGRAM_API_KEY


def seed_memory(text, user_id, **kwargs):
    """Add a memory and block until the async pipeline commits it."""
    run = client.memories.add(text, user_id=user_id, **kwargs)
    status = client.runs.wait(run.run_id, timeout=60.0)
    print(f'run {run.run_id}: {status.status} '
          f'(+{len(status.memories_created)} / ~{len(status.memories_updated)})')
    return status


def wait_until_searchable(query, user_id, needle, timeout=60.0, **kwargs):
    """Poll search until `needle` shows up (handles eventual consistency)."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        results = client.memories.search(query=query, user_id=user_id, **kwargs)
        if any(needle.lower() in m.content.lower() for m in results):
            return results
        time.sleep(2.0)
    return client.memories.search(query=query, user_id=user_id, **kwargs)

In [ ]:
USER = 'playground-user'

seed_memory('The user is learning to play the cello.', user_id=USER)

results = client.memories.search(query='hobbies', user_id=USER)
for m in results:
    print(f'- ({m.score}) {m.content}')

If you see the cello memory above, your setup works. ✅

Move on to:
- `1_middleware_agent_memory.ipynb` — drop-in agent memory
- `2_memory_tools.ipynb` — agent-controlled memory tools
- `3_engram_store.ipynb` — the LangGraph `BaseStore` adapter